# Where do critics and audiences disagree, and can we predict it?

**Research question 2** (replacing the genre-only question in `main.ipynb`): which film attributes are associated with the gap between the Tomatometer and the audience score, and can a model predict, for films it hasn't seen, which ones will split critics and audiences by 20 or more points?

**Data:** the 2,454 Rotten Tomatoes films with both scores. The gap is **Tomatometer − audience score**, in points on a 0–100 scale; positive means critics liked the film more.

**What the models may use.** The gap is built from the two scores, so anything that tracks either score leaks the answer: average critic rating, top-critic score, consensus text, sentiment labels. None of these are used. Features are grouped by when they're known:

| Group | Features | Known |
|---|---|---|
| Pre-release | genre, content rating, runtime, release year and month, original language, plot synopsis | before any review |
| Post-release | number of critic and audience ratings (popularity) | only after release |

The main models use pre-release features only, so they answer a question a studio could ask before reviews arrive. Popularity is added separately and labeled as such.

**Evaluation:** 5-fold cross-validation repeated twice (10 folds), with each model's settings picked by an inner 3-fold CV inside each training fold (nested CV). Imputation, scaling and text vectorizers are fit inside each fold.

## Setup

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.inspection import partial_dependence
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import (GridSearchCV, KFold, RepeatedKFold, RepeatedStratifiedKFold,
                                     StratifiedKFold, cross_val_predict, cross_validate)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from story_reception.features import (LEAKY_COLUMNS, content_rating_group, feature_family, post_release_features,
                                      pre_release_features)

pd.set_option('display.max_colwidth', 120)
SEED = 42
CLEAN_DIR, CACHE_DIR, FIG_DIR = 'data/cleaned', 'data/cache', 'figures'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Chart palette (same as main.ipynb)
SURFACE, INK, INK2, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#e6e5e1'
POS, NEG, OLD = '#2a78d6', '#e34948', '#b8b7b2'     # blue = critics higher, red = audiences higher
SERIES = ['#2a78d6', '#eb6834', '#1baf7a']
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'axes.edgecolor': GRID, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.axisbelow': True,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'xtick.color': INK2, 'ytick.color': INK2, 'text.color': INK,
    'font.size': 10, 'legend.frameon': False, 'figure.dpi': 110,
})

In [ ]:
movies = pd.read_csv(f'{CLEAN_DIR}/cleaned_rotten_tomatoes_movies.csv')
films = movies.dropna(subset=['tomatometer', 'audience_score']).reset_index(drop=True)
gap = (films['tomatometer'] - films['audience_score']).to_numpy()
big_split = (np.abs(gap) >= 20).astype(int)

X_pre = pre_release_features(films)
X_post = post_release_features(films)
synopsis = films['synopsis'].fillna('').rename('synopsis')
assert not set(X_pre.columns) & set(LEAKY_COLUMNS)

print(f'{len(films):,} films; gap mean {gap.mean():.1f}, sd {gap.std():.1f}; '
      f'{big_split.mean():.0%} split by 20+ points '
      f'({(gap >= 20).mean():.0%} critics higher, {(gap <= -20).mean():.0%} audiences higher)')
print(f'{X_pre.shape[1]} pre-release features, {X_post.shape[1]} post-release; '
      f'{(synopsis.str.len() == 0).sum()} films without a synopsis')

## 1. What the gap looks like

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
bins = np.arange(-100, 105, 5)
ax.hist(gap[gap <= -20], bins=bins, color=NEG, rwidth=0.88, label='audiences 20+ higher')
ax.hist(gap[(gap > -20) & (gap < 20)], bins=bins, color=OLD, rwidth=0.88, label='within 20 points')
ax.hist(gap[gap >= 20], bins=bins, color=POS, rwidth=0.88, label='critics 20+ higher')
ax.axvline(0, color=INK2, lw=1)
ax.set_xlabel('Gap: Tomatometer − audience score (points)')
ax.set_ylabel('Films')
ax.set_title('Critics and audiences disagree by 20+ points on 46% of films', pad=8)
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

Average gap by group, with 95% intervals (mean ± 1.96 standard errors). Blue groups lean toward critics, red toward audiences, and gray intervals include zero. Groups with fewer than 15 films are left out.

In [ ]:
def group_gap(labels, order=None, min_n=15):
    d = pd.DataFrame({'group': labels, 'gap': gap}).dropna()
    g = d.groupby('group')['gap'].agg(['mean', 'std', 'count'])
    g = g[g['count'] >= min_n]
    g['se'] = g['std'] / np.sqrt(g['count'])
    return g.loc[[o for o in order if o in g.index]] if order else g.sort_values('mean')

runtime_bin = pd.cut(films['runtime_min'], [0, 89, 109, 129, 1000], labels=['under 90 min', '90–109', '110–129', '130+'])
era = pd.cut(films['release_year'], [0, 1999, 2014, 2019, 2023, 2100],
             labels=['before 2000', '2000–14', '2015–19', '2020–23', '2024–26'])
lang = films['original_language'].fillna('Unknown')
lang = lang.where(lang.isin(lang.value_counts().index[:8]), 'other')
popularity = pd.qcut(np.log1p(films['audience_review_count'].fillna(0)), 5,
                     labels=['least rated fifth', '2nd', '3rd', '4th', 'most rated fifth'])

panels = [('Content rating', group_gap(content_rating_group(films['content_rating']),
                                       ['G', 'PG', 'PG-13', 'R', 'TV', 'Unrated'])),
          ('Runtime', group_gap(runtime_bin.astype(str).where(runtime_bin.notna()), list(runtime_bin.cat.categories))),
          ('Release era', group_gap(era.astype(str).where(era.notna()), list(era.cat.categories))),
          ('Original language', group_gap(lang)),
          ('Audience ratings collected (post-release)', group_gap(popularity.astype(str), list(popularity.cat.categories)))]

fig, axes = plt.subplots(len(panels), 1, figsize=(8, 11), sharex=True,
                         gridspec_kw={'height_ratios': [len(p[1]) + 1 for p in panels]})
for ax, (title, g) in zip(axes, panels):
    lo, hi = g['mean'] - 1.96 * g['se'], g['mean'] + 1.96 * g['se']
    colors = np.where((lo > 0), POS, np.where(hi < 0, NEG, OLD))
    y = np.arange(len(g))[::-1]
    ax.hlines(y, lo, hi, color=colors, lw=2)
    ax.scatter(g['mean'], y, color=colors, s=40, zorder=3, edgecolor=SURFACE, linewidth=1.5)
    ax.set_yticks(y, [f'{k}  ({int(n):,})' for k, n in zip(g.index, g['count'])])
    ax.axvline(0, color=INK2, lw=1)
    ax.set_title(title, fontsize=11)
    ax.grid(axis='y', visible=False)
axes[-1].set_xlabel('Average gap, critic − audience (points, 95% interval)')
fig.suptitle('What goes with critics or audiences rating a film higher', x=0.01, ha='left',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/gap-by-group.png', dpi=160, bbox_inches='tight')
plt.show()

*(eda: filled in after results)*

## 2. Synopsis features

The plot synopsis is the one text field that exists before release. Two versions:
- **TF-IDF** of words and two-word phrases, fit inside each CV fold.
- **Sentence embeddings** from a pretrained model (`all-MiniLM-L6-v2`, 384 dimensions), computed once. The model was trained on general web text, not on these films or their scores, so precomputing them leaks nothing. If `sentence-transformers` isn't installed, the embedding rows are skipped.

In [ ]:
EMB_PATH = f'{CACHE_DIR}/synopsis-embeddings-minilm.npz'
try:
    from sentence_transformers import SentenceTransformer
    HAVE_EMB = True
except ImportError:
    HAVE_EMB = False

if HAVE_EMB:
    cached = np.load(EMB_PATH, allow_pickle=True) if os.path.exists(EMB_PATH) else None
    if cached is not None and list(cached['urls']) == films['movie_url'].tolist():
        emb = cached['emb']
        print('Loaded cached embeddings', emb.shape)
    else:
        start = time.time()
        encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        emb = encoder.encode(synopsis.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
        np.savez(EMB_PATH, urls=np.array(films['movie_url']), emb=emb)
        print(f'Encoded {len(emb):,} synopses in {time.time() - start:.0f}s', emb.shape)
    E = pd.DataFrame(emb, columns=[f'emb {i}' for i in range(emb.shape[1])], index=films.index)
else:
    E = pd.DataFrame(index=films.index)
    print('sentence-transformers not installed: embedding models are skipped')

data = pd.concat([X_pre, X_post, E, synopsis], axis=1)
GENRE = list(X_pre.filter(like='genre: ').columns)
PRE, POST, EMB = list(X_pre.columns), list(X_post.columns), list(E.columns)

## 3. Predicting the gap

Every model is scored on the same 10 outer folds. R² is the share of gap variance explained on held-out films, and MAE is the typical miss in points.

In [ ]:
outer = RepeatedKFold(n_splits=5, n_repeats=2, random_state=SEED)
inner = KFold(n_splits=3, shuffle=True, random_state=SEED)

def numeric(cols, scale=True):
    steps = [SimpleImputer(strategy='median', add_indicator=True)] + ([StandardScaler()] if scale else [])
    return ('num', make_pipeline(*steps), cols)

def tfidf():
    return ('text', TfidfVectorizer(min_df=3, max_features=5000, ngram_range=(1, 2), stop_words='english',
                                    sublinear_tf=True), 'synopsis')

def ridge(cols, text=False):
    parts = [numeric(cols)] + ([tfidf()] if text else [])
    pipe = make_pipeline(ColumnTransformer(parts), Ridge())
    return GridSearchCV(pipe, {'ridge__alpha': [1, 10, 30, 100, 300, 1000]}, cv=inner, scoring='r2')

def boosting(cols):
    pipe = make_pipeline(ColumnTransformer([('num', 'passthrough', cols)]),
                         HistGradientBoostingRegressor(max_iter=400, l2_regularization=1.0, min_samples_leaf=20,
                                                       early_stopping=True, validation_fraction=0.15,
                                                       n_iter_no_change=20, random_state=SEED))
    grid = {'histgradientboostingregressor__learning_rate': [0.03, 0.1],
            'histgradientboostingregressor__max_leaf_nodes': [7, 15]}
    return GridSearchCV(pipe, grid, cv=inner, scoring='r2')

def forest(cols):
    return make_pipeline(ColumnTransformer([numeric(cols, scale=False)]),
                         RandomForestRegressor(n_estimators=500, min_samples_leaf=5, max_features=0.33,
                                               n_jobs=-1, random_state=SEED))

REG_MODELS = {   # name: (model, feature group label)
    'Mean baseline':                       (DummyRegressor(), 'none'),
    'Ridge: genre only':                   (ridge(GENRE), 'pre-release'),
    'Ridge: pre-release metadata':         (ridge(PRE), 'pre-release'),
    'Random forest: metadata':             (forest(PRE), 'pre-release'),
    'Gradient boosting: metadata':         (boosting(PRE), 'pre-release'),
    'Ridge: metadata + synopsis TF-IDF':   (ridge(PRE, text=True), 'pre-release'),
}
if HAVE_EMB:
    REG_MODELS['Ridge: metadata + synopsis embeddings'] = (ridge(PRE + EMB), 'pre-release')
    REG_MODELS['Gradient boosting: metadata + embeddings'] = (boosting(PRE + EMB), 'pre-release')
REG_MODELS['Ridge: metadata + popularity'] = (ridge(PRE + POST), 'post-release')
REG_MODELS['Gradient boosting: metadata + popularity'] = (boosting(PRE + POST), 'post-release')

reg_rows = []
for name, (model, group) in REG_MODELS.items():
    start = time.time()
    s = cross_validate(model, data, gap, cv=outer, scoring={'r2': 'r2', 'mae': 'neg_mean_absolute_error'})
    reg_rows.append({'model': name, 'features known': group, 'R² mean': s['test_r2'].mean(),
                     'R² sd': s['test_r2'].std(), 'MAE (points)': -s['test_mae'].mean(),
                     'seconds': round(time.time() - start)})
    print(f'{name:45s} R² {s["test_r2"].mean():.3f} ± {s["test_r2"].std():.3f}   ({time.time() - start:.0f}s)')
reg_results = pd.DataFrame(reg_rows).set_index('model')
display(reg_results.style.format({'R² mean': '{:.3f}', 'R² sd': '{:.3f}', 'MAE (points)': '{:.1f}'}))

In [ ]:
order = reg_results.index[::-1]
colors = [OLD if g == 'none' else (SERIES[1] if g == 'post-release' else SERIES[0])
          for g in reg_results.loc[order, 'features known']]
fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(order) + 1.4))
y = np.arange(len(order))
ax.hlines(y, reg_results.loc[order, 'R² mean'] - reg_results.loc[order, 'R² sd'],
          reg_results.loc[order, 'R² mean'] + reg_results.loc[order, 'R² sd'], color=colors, lw=2)
ax.scatter(reg_results.loc[order, 'R² mean'], y, color=colors, s=46, zorder=3, edgecolor=SURFACE, linewidth=1.5)
for yv, v in zip(y, reg_results.loc[order, 'R² mean']):
    ax.text(v, yv + 0.28, f'{v:.2f}', ha='center', fontsize=8, color=INK2)
ax.set_yticks(y, order)
ax.axvline(0, color=INK2, lw=1)
ax.grid(axis='y', visible=False)
ax.set_xlabel('Held-out R² (mean ± sd over 10 folds)')
ax.set_title('Predicting the critic–audience gap', pad=20)
ax.text(0, 1.01, 'Blue: pre-release features only. Orange: adds post-release popularity.',
        transform=ax.transAxes, color=INK2, fontsize=9, va='bottom')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/gap-models.png', dpi=160, bbox_inches='tight')
plt.show()

*(regression: filled in after results)*

### How much of the gap is noise?

Many films have very few ratings: 40% have fewer than 50 audience ratings, and some audience scores rest on 3 or 4. A score built from *n* ratings has sampling error, so part of every film's gap is noise that no model could predict. Treating each score as a share of positive ratings, its sampling variance is roughly p(1 − p) / n, in points² × 10,000. Summing the critic and audience parts gives each film's expected noise variance. The ratio of the average noise variance to the gap's total variance estimates the unpredictable share and puts a ceiling on R².

In [ ]:
n_c = films['tomatometer_count'].to_numpy(dtype=float)
n_a = films['audience_review_count'].to_numpy(dtype=float)
p_c, p_a = films['tomatometer'].to_numpy() / 100, films['audience_score'].to_numpy() / 100
counted = (n_c > 0) & (n_a > 0)
noise_var = 1e4 * (np.clip(p_c * (1 - p_c), 0.05, None) / n_c + np.clip(p_a * (1 - p_a), 0.05, None) / n_a)

rows = []
for label, mask in [('all films with rating counts', counted),
                    ('50+ audience and 5+ critic ratings', counted & (n_a >= 50) & (n_c >= 5)),
                    ('200+ audience and 20+ critic ratings', counted & (n_a >= 200) & (n_c >= 20))]:
    share = noise_var[mask].mean() / gap[mask].var()
    rows.append({'films': label, 'n': int(mask.sum()), 'gap sd (points)': gap[mask].std(),
                 'noise share of variance': share, 'R² ceiling': 1 - share,
                 'split by 20+': (np.abs(gap[mask]) >= 20).mean()})
display(pd.DataFrame(rows).set_index('films').style.format(
    {'gap sd (points)': '{:.1f}', 'noise share of variance': '{:.0%}', 'R² ceiling': '{:.2f}', 'split by 20+': '{:.0%}'}))

**Robustness check:** the main pre-release models refit on the films with at least 50 audience and 5 critic ratings, where sampling noise is small. Same nested CV.

In [ ]:
reliable = counted & (n_a >= 50) & (n_c >= 5)
rel_rows = []
for name in ['Mean baseline', 'Ridge: genre only', 'Ridge: pre-release metadata', 'Random forest: metadata',
             'Ridge: metadata + synopsis TF-IDF', 'Gradient boosting: metadata + popularity']:
    s = cross_validate(REG_MODELS[name][0], data[reliable], gap[reliable], cv=outer,
                       scoring={'r2': 'r2', 'mae': 'neg_mean_absolute_error'})
    rel_rows.append({'model': name, 'all films R²': reg_results.loc[name, 'R² mean'],
                     'reliable films R²': s['test_r2'].mean(), 'sd': s['test_r2'].std(),
                     'reliable MAE (points)': -s['test_mae'].mean()})
print(f'{reliable.sum():,} reliable films (of {len(films):,})')
display(pd.DataFrame(rel_rows).set_index('model').style.format(
    {'all films R²': '{:.3f}', 'reliable films R²': '{:.3f}', 'sd': '{:.3f}', 'reliable MAE (points)': '{:.1f}'}))

*(noise: filled in after results)*

## 4. Flagging films likely to split critics and audiences

A yes/no version: will the film's critic and audience scores differ by **20 or more points** (either direction)? 46% of films do, so the useful question is whether the model's most confident picks are right more often than that. Two metrics:
- **ROC-AUC:** the chance a random split film is ranked above a random non-split film (0.5 is guessing).
- **Precision in the top 10%:** of the 10% of films the model is most sure will split, how many actually do (vs. a 46% base rate).

Scored by 5-fold stratified CV repeated twice; logistic regression's penalty is tuned by an inner CV.

In [ ]:
tie_rng = np.random.default_rng(SEED)

def top_decile_precision(y_true, prob):
    k = max(1, int(round(0.1 * len(prob))))
    order = np.lexsort((tie_rng.random(len(prob)), -prob))   # highest first; ties broken at random
    return y_true[order[:k]].mean()

def evaluate_classifier(model, X, y, cv):
    aucs, precs = [], []
    for tr, te in cv.split(X, y):
        m = clone(model).fit(X.iloc[tr], y[tr])
        prob = m.predict_proba(X.iloc[te])[:, 1]
        aucs.append(roc_auc_score(y[te], prob))
        precs.append(top_decile_precision(y[te], prob))
    return np.mean(aucs), np.std(aucs), np.mean(precs)

def logistic(cols, text=False):
    parts = [numeric(cols)] + ([tfidf()] if text else [])
    pipe = make_pipeline(ColumnTransformer(parts), LogisticRegression(max_iter=3000))
    return GridSearchCV(pipe, {'logisticregression__C': [0.01, 0.1, 1]}, cv=StratifiedKFold(3, shuffle=True,
                                                                                          random_state=SEED),
                        scoring='roc_auc')

def boosting_clf(cols):
    return make_pipeline(ColumnTransformer([('num', 'passthrough', cols)]),
                         HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                                        min_samples_leaf=20, l2_regularization=1.0,
                                                        early_stopping=True, validation_fraction=0.15,
                                                        n_iter_no_change=20, random_state=SEED))

CLF_MODELS = {
    'Base rate':                              (DummyClassifier(strategy='prior'), 'none'),
    'Logistic: pre-release metadata':         (logistic(PRE), 'pre-release'),
    'Logistic: metadata + synopsis TF-IDF':   (logistic(PRE, text=True), 'pre-release'),
    'Gradient boosting: metadata':            (boosting_clf(PRE), 'pre-release'),
}
if HAVE_EMB:
    CLF_MODELS['Logistic: metadata + embeddings'] = (logistic(PRE + EMB), 'pre-release')
CLF_MODELS['Gradient boosting: metadata + popularity'] = (boosting_clf(PRE + POST), 'post-release')

cv_c = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
clf_rows = []
for name, (model, group) in CLF_MODELS.items():
    auc, auc_sd, prec = evaluate_classifier(model, data, big_split, cv_c)
    clf_rows.append({'model': name, 'features known': group, 'ROC-AUC': auc, 'AUC sd': auc_sd,
                     'precision in top 10%': prec})
    print(f'{name:42s} AUC {auc:.3f} ± {auc_sd:.3f}   top-10% precision {prec:.0%}')
clf_results = pd.DataFrame(clf_rows).set_index('model')
display(clf_results.style.format({'ROC-AUC': '{:.3f}', 'AUC sd': '{:.3f}', 'precision in top 10%': '{:.0%}'}))

*(classification: filled in after results)*

## 5. What drives the predictions

**Feature-family importance:** for the best pre-release model, the drop in held-out R² when one family of features is shuffled (all genre flags together, all content-rating flags together, and so on). Shuffling breaks the link between that family and the gap while keeping everything else, so a bigger drop means the model relies on it more. Measured on 5 held-out folds, 5 shuffles each.

In [ ]:
pre_models = reg_results[reg_results['features known'] == 'pre-release']
best_name = pre_models['R² mean'].idxmax()
best_model = REG_MODELS[best_name][0]
print('Best pre-release model:', best_name)

def model_columns(model):
    # The input columns a (possibly grid-searched) pipeline reads, from its ColumnTransformer
    pipe = model.estimator if isinstance(model, GridSearchCV) else model
    ct = next(step for step in pipe.named_steps.values() if isinstance(step, ColumnTransformer))
    cols = []
    for _, _, c in ct.transformers:
        cols += [c] if isinstance(c, str) else list(c)
    return cols

families = {}
for c in model_columns(best_model):
    families.setdefault('synopsis' if c.startswith('emb ') or c == 'synopsis' else feature_family(c), []).append(c)

rng = np.random.default_rng(SEED)
drops = {f: [] for f in families}
for tr, te in KFold(n_splits=5, shuffle=True, random_state=SEED).split(data):
    m = clone(best_model).fit(data.iloc[tr], gap[tr])
    base = r2_score(gap[te], m.predict(data.iloc[te]))
    for fam, cols in families.items():
        for _ in range(5):
            shuffled = data.iloc[te].copy()
            perm = rng.permutation(len(te))
            shuffled[cols] = shuffled[cols].to_numpy()[perm]     # shuffle the family's columns together
            drops[fam].append(base - r2_score(gap[te], m.predict(shuffled)))
importance = pd.DataFrame({'R² drop': {f: np.mean(v) for f, v in drops.items()},
                           'sd': {f: np.std(v) for f, v in drops.items()}}).sort_values('R² drop')

fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(importance) + 1.3))
y = np.arange(len(importance))
ax.hlines(y, importance['R² drop'] - importance['sd'], importance['R² drop'] + importance['sd'], color=SERIES[0], lw=2)
ax.scatter(importance['R² drop'], y, color=SERIES[0], s=46, zorder=3, edgecolor=SURFACE, linewidth=1.5)
ax.set_yticks(y, importance.index)
ax.axvline(0, color=INK2, lw=1)
ax.grid(axis='y', visible=False)
ax.set_xlabel('Drop in held-out R² when the family is shuffled (mean ± sd)')
ax.set_title(f'Which features the model relies on', pad=20)
ax.text(0, 1.01, best_name, transform=ax.transAxes, color=INK2, fontsize=9, va='bottom')
plt.tight_layout()
plt.show()

**Direction of the main effects.** The Ridge metadata model's standardized coefficients, i.e. the change in predicted gap (points) for a one-standard-deviation change in each feature, holding the others fixed. These are fit on all films and are associations, not effects.

In [ ]:
meta = clone(REG_MODELS['Ridge: pre-release metadata'][0]).fit(data, gap)
ridge_step = meta.best_estimator_.named_steps['ridge']
names = meta.best_estimator_.named_steps['columntransformer'].get_feature_names_out()
coefs = pd.Series(ridge_step.coef_, index=[n.replace('num__', '') for n in names])
coefs = coefs[~coefs.index.str.startswith('missingindicator')]
show = pd.concat([coefs.nsmallest(10), coefs.nlargest(10)]).drop_duplicates().sort_values()
fig, ax = plt.subplots(figsize=(7.5, 0.3 * len(show) + 1.3))
ax.barh(show.index, show, color=np.where(show > 0, POS, NEG), height=0.65)
ax.axvline(0, color=INK2, lw=1)
ax.grid(axis='y', visible=False)
ax.set_xlabel('Change in predicted gap per 1 sd of the feature (points)')
ax.set_title('Ridge on pre-release metadata: strongest associations', pad=20)
ax.text(0, 1.01, f"alpha = {meta.best_params_['ridge__alpha']}. Blue: critics higher. Red: audiences higher.",
        transform=ax.transAxes, color=INK2, fontsize=9, va='bottom')
plt.tight_layout()
plt.show()

**Runtime, holding everything else fixed.** Partial dependence from the gradient-boosting metadata model: the average predicted gap if every film had a given runtime.

In [ ]:
gb = clone(REG_MODELS['Gradient boosting: metadata'][0]).fit(data, gap)
pd_runtime = partial_dependence(gb.best_estimator_, data, ['runtime (min)'], grid_resolution=40,
                                percentiles=(0.05, 0.95))
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(pd_runtime['grid_values'][0], pd_runtime['average'][0], color=SERIES[0], lw=2)
ax.axhline(0, color=INK2, lw=1)
ax.set_xlabel('Runtime (minutes)')
ax.set_ylabel('Predicted gap (points)')
ax.set_title('Shorter films lean toward critics')
plt.tight_layout()
plt.show()

*(drivers: filled in after results)*

## 6. Case studies

Out-of-fold predictions (each film predicted by a model that never saw it) from the best pre-release model. Three films where the model saw a big split coming, and three big splits it missed.

In [ ]:
oof = cross_val_predict(best_model, data, gap, cv=KFold(n_splits=5, shuffle=True, random_state=SEED))
cases = films[['title', 'release_year', 'content_rating', 'runtime_min', 'original_language', 'genres',
               'tomatometer', 'tomatometer_count', 'audience_score', 'audience_review_count']].assign(
    gap=gap, predicted=oof.round(1)).rename(columns={'tomatometer_count': 'critic ratings',
                                                     'audience_review_count': 'audience ratings'})
cases['genres'] = cases['genres'].str.replace('|', ', ', regex=False)
caught = cases[(np.abs(cases['gap']) >= 40) & (np.sign(cases['gap']) == np.sign(cases['predicted']))]
caught = caught.reindex(caught['predicted'].abs().sort_values(ascending=False).index).head(3)
missed = cases[np.abs(cases['gap']) >= 50].assign(miss=lambda d: (d['gap'] - d['predicted']).abs())
missed = missed.sort_values('miss', ascending=False).head(3).drop(columns='miss')
print('Saw it coming:')
display(caught)
print('Missed:')
display(missed)
print(f"Out-of-fold R² {r2_score(gap, oof):.3f}, MAE {mean_absolute_error(gap, oof):.1f} points")

*(cases: filled in after results)*

## Conclusion
*(conclusion: filled in after results)*